# Supplementary Tables S3-S5: Descriptive statistics and top countries of the citation preference networks

This notebook computes the descriptive statistics of the positive and negative networks (Table S3) and the top 10 countries by in-degree and PageRank centrality in the positive (Table S4) and negative (Table S5) networks, using the cumulative network of 1990-2017.

It reads the processed data in the data folder (`path2data`) and can be run on its own.

In [1]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm

import matplotlib.pylab as plt
import matplotlib.gridspec as gridspec
import matplotlib.transforms as mtransforms
import seaborn as sns

from vizutils import pvalue2stars, table_row, make_multinomial_latex_table, logistic_coefficient_plot

import pyscisci.all as pyscisci

# set this path to where the OpenAlex database is stored
path2openalex = '/Users/hgt6rn/Documents/DataSets/OpenAlex'

# set this path to where the processed data (Source Data) is stored:
path2sourcedata = "Source Data"
# when the notebook is run from the code folder, the Source Data folder is one level up
if not os.path.exists(path2sourcedata):
    path2sourcedata = os.path.join("..", "Source Data")

# the processed input data are in the data folder, next to the Source Data folder
path2data = os.path.join(os.path.dirname(os.path.abspath(path2sourcedata)), 'data')

Please install python-Levenshtein to take full advantage of the NLP tools. 
 pip install python-Levenshtein
Please install sparse_dot_topn to take full advantage of the NLP tools. 
 pip install sparse_dot_topn


/Users/psp2nq/anaconda3/lib/python3.11/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


# The cumulative citation preference network

In [2]:
# lets focus on the citation without the same organization and self citations

country_auc_df = pd.read_csv(os.path.join(path2data, 'oa_countrycites_nosameorg_auc.csv.gz'))

country_auc_df = country_auc_df[country_auc_df['N']>=50]

country_auc_df['Zscore'] = (country_auc_df['AUC'] - 0.5)/np.sqrt(country_auc_df['Cov'])
# one-sided test in the direction of the observed preference: H1 is AUC > 0.5 (over-citation)
# when Z > 0, and AUC < 0.5 (under-citation) when Z < 0
country_auc_df['pvalue'] = spstats.norm.sf(np.abs(country_auc_df['Zscore'].values))

def reject_holm(pvalues):
    reject, pvalues, _, _ = holm(pvalues, alpha=0.01, method='hs', is_sorted=False, returnsorted=False)
    return reject
country_auc_df['Significant'] = country_auc_df.groupby('CitedYear')['pvalue'].transform(reject_holm)

net_auc = country_auc_df[country_auc_df['CitingCountry'] != country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
net_auc = net_auc[net_auc['Significant']]
self_auc = country_auc_df[country_auc_df['CitingCountry'] == country_auc_df['CitedCountry']].sort_values(by=['CitedYear', 'CitedCountry']).reset_index(drop=True)
self_auc = self_auc[self_auc['Significant']]

syear, eyear = 1990, 2017
net_auc = net_auc[net_auc['CitedYear'] >= syear]
net_auc = net_auc[net_auc['CitedYear'] <= eyear]

comm2country = {0:['AE', 'BD', 'CN', 'CY', 'DZ', 'EG', 'ID', 'IN', 'IQ', 'IR', 'JO', 'KH', 'KR', 'KW', 'MA', 'MY', 'OM', 'PK', 'QA', 'SA', 'SG', 'TH', 'TN', 'TR', 'TW', 'VN'],
1:['AR', 'BN', 'BO', 'BR', 'BW', 'BY', 'CL', 'CM', 'CO', 'CR', 'GE', 'HK', 'HR', 'LU', 'MZ', 'NC', 'PA', 'PE', 'RU', 'SJ', 'ST', 'UY', 'VE'],
2:['AT', 'BE', 'CA', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HU', 'IE', 'IL', 'IS', 'IT', 'JP', 'MX', 'NL', 'NO', 'NP', 'NZ', 'PH', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK'],
3:['AU', 'CH', 'GB', 'US'],
4:['BH', 'ET', 'GH', 'KE', 'MW', 'NG', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']}

country2comm = {c:ci for ci,clist in comm2country.items() for c in clist}
poscountrylist = sorted(list(country2comm.keys()))

connected_countries = np.sort(np.unique(net_auc[net_auc['Zscore'] > 0][['CitingCountry', 'CitedCountry']].values.flatten()))

# the cumulative network: each country pair keeps its most recent significant citation preference
cumulative_net = net_auc.sort_values('CitedYear').drop_duplicates(subset=['CitingCountry', 'CitedCountry'], keep='last')

networks = {}
for name, sub in [('Positive', cumulative_net[cumulative_net['Zscore'] > 0]), ('Negative', cumulative_net[cumulative_net['Zscore'] < 0])]:
    g = nx.DiGraph()
    g.add_edges_from(sub[['CitingCountry', 'CitedCountry']].values)
    networks[name] = g

## Supplementary Table S3: descriptive statistics

In [3]:
summary = []
pageranks = {}
for name, g in networks.items():
    pageranks[name] = pd.Series(nx.pagerank(g, alpha=0.85))
    # the average in-degree equals the average out-degree (edges / nodes)
    summary.append([name + ' Network', g.number_of_nodes(), g.number_of_edges(), nx.density(g),
                    nx.number_weakly_connected_components(g), g.number_of_edges()/g.number_of_nodes(),
                    pageranks[name].mean()])
summary = pd.DataFrame(summary, columns=['Network', 'Nodes', 'Edges', 'Density', 'Components (weakly connected)',
                                         'Avg In/Out-Degree', 'Avg PageRank'])
summary.round(2)

,Network,Nodes,Edges,Density,Components (weakly connected),Avg In/Out-Degree,Avg PageRank
0,Positive Network,93,536,0.06,1,5.76,0.01
1,Negative Network,145,1471,0.07,1,10.14,0.01


## Supplementary Tables S4 and S5: top 10 countries by in-degree and PageRank centrality

In [4]:
top_tables = {}
for name, g in networks.items():
    indegree = pd.DataFrame(list(g.in_degree()), columns=['Country(In-degree)', 'In-degree'])
    # ties in in-degree are listed alphabetically
    indegree = indegree.sort_values(['In-degree', 'Country(In-degree)'], ascending=[False, True]).head(10).reset_index(drop=True)
    pagerank = pageranks[name].sort_values(ascending=False).head(10).reset_index()
    pagerank.columns = ['Country(PageRank)', 'PageRank centrality']
    top = pd.concat([indegree, pagerank], axis=1)
    top.insert(0, 'Ranking', np.arange(1, 11))
    top_tables[name] = top
    print(name)
    print(top.round(3).to_string(index=False))

Positive
 Ranking Country(In-degree)  In-degree Country(PageRank)  PageRank centrality
       1                 CH         36                CH                0.052
       2                 GB         31                GB                0.046
       3                 DE         22                AU                0.045
       4                 NL         22                US                0.036
       5                 BE         20                NL                0.034
       6                 FR         20                DE                0.033
       7                 DK         19                SA                0.032
       8                 SA         19                DK                0.031
       9                 US         19                SE                0.030
      10                 AU         17                BE                0.030
Negative
 Ranking Country(In-degree)  In-degree Country(PageRank)  PageRank centrality
       1                 CN         86        

## Source Data for Supplementary Tables 3-5

The values reported in the tables, written to `Source Data/SourceData_3_TableS3.xlsx`, `SourceData_4_TableS4.xlsx` and `SourceData_5_TableS5.xlsx`.

In [5]:
# the Source Data helpers (sourcedatautils.py, in the code folder like vizutils.py)
from sourcedatautils import write_source_data

network_description = ('Cumulative international citation preference network (1990-2017): each directed edge from the citing '
                       'country to the cited country is the most recent statistically significant citation preference '
                       '(N >= 50, Holm-Sidak correction with alpha = 0.01 within each year); positive edges have Z-score > 0 '
                       'and negative edges Z-score < 0. PageRank centrality uses a damping factor of 0.85.')

write_source_data('SourceData_3_TableS3.xlsx',
                  'Supplementary Table 3: Descriptive statistics of the positive and negative preference networks.',
                  network_description + ' Components are weakly connected components.',
                  [('Network statistics', 'Descriptive statistics of each network.', summary)])

for fname, title, name in [('SourceData_4_TableS4.xlsx', 'Supplementary Table 4: Top 10 countries by in-degree and PageRank '
                            'centrality in the positive preference network.', 'Positive'),
                           ('SourceData_5_TableS5.xlsx', 'Supplementary Table 5: Top 10 countries by in-degree and PageRank '
                            'centrality in the negative preference network.', 'Negative')]:
    g = networks[name]
    nodes = pd.DataFrame({'Country':list(g.nodes())})
    nodes['In-degree'] = nodes['Country'].map(dict(g.in_degree()))
    nodes['PageRank centrality'] = nodes['Country'].map(pageranks[name])
    nodes = nodes.sort_values('Country').reset_index(drop=True)
    write_source_data(fname, title, network_description + ' Ties in in-degree are listed alphabetically.',
                      [('Top 10', 'Top 10 countries by in-degree and by PageRank centrality.', top_tables[name]),
                       ('All countries', 'In-degree and PageRank centrality of every country in the network.', nodes)])

wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_3_TableS3.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_4_TableS4.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/SourceData_5_TableS5.xlsx
wrote /Users/psp2nq/Documents/NationalBiasCode/Source Data/README.txt
